# 02 — EDA and Preprocessing

This notebook explores the raw data and engineers all features used by the model:
- **Exploratory data analysis** — distributions, correlations, outliers
- **Feature engineering** — MA, RSI, MACD, and other technical indicators via `features.py`
- **Scaling and sequence building** — MinMaxScaler + 60-day lookback windows

All transformed data is saved to `data/processed/` ready for notebook 03.

## 1. Imports

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from data_loader import (
    TICKERS, TRAIN_START, TRAIN_END,
    download_data, load_macro_data
)
from features import engineer_features
from model import FEATURES, SEQUENCE_LENGTH

print('All imports successful')
print(f'Sequence length : {SEQUENCE_LENGTH} days')
print(f'Feature count   : {len(FEATURES)}')
print(f'Features        : {FEATURES}')

## 2. Load Raw Data

Load from CSV if already downloaded, otherwise fetch from Yahoo Finance.

In [ ]:
etf_data = {}

for ticker in TICKERS:
    csv_path = f"data/raw/{ticker.replace('.', '_')}_train.csv"
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
        print(f'Loaded {ticker} from CSV ({len(df)} rows)')
    else:
        print(f'Downloading {ticker} from Yahoo Finance...')
        df = download_data(ticker, TRAIN_START, TRAIN_END)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
    etf_data[ticker] = df

macro_csv = 'data/raw/macro_train.csv'
if os.path.exists(macro_csv):
    macro = pd.read_csv(macro_csv, index_col=0, parse_dates=True)
    print(f'Loaded macro data from CSV ({len(macro)} rows)')
else:
    print('Downloading macro data...')
    macro = load_macro_data(TRAIN_START, TRAIN_END)

print('\nDone.')

## 3. Closing Price Distributions

Histograms show how prices are distributed across the training window — useful for spotting skew before scaling.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
axes = axes.flatten()
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for i, (ticker, df) in enumerate(etf_data.items()):
    axes[i].hist(df['Close'], bins=40, color=colors[i], alpha=0.75, edgecolor='white')
    axes[i].axvline(df['Close'].mean(), color='black', linestyle='--', linewidth=1, label=f'Mean: ${df["Close"].mean():.2f}')
    axes[i].set_title(f'{ticker} — Close Price Distribution')
    axes[i].set_xlabel('Price (AUD)')
    axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Closing Price Distributions — Training Data (2022–2024)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Daily Returns Analysis

Daily returns (percentage change) reveal volatility and whether returns are approximately normally distributed.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
axes = axes.flatten()

for i, (ticker, df) in enumerate(etf_data.items()):
    returns = df['Close'].pct_change().dropna() * 100
    axes[i].hist(returns, bins=50, color=colors[i], alpha=0.75, edgecolor='white')
    axes[i].axvline(0, color='black', linestyle='--', linewidth=1)
    axes[i].set_title(f'{ticker} — Daily Returns')
    axes[i].set_xlabel('Daily Return (%)')
    axes[i].set_ylabel('Frequency')
    std = returns.std()
    axes[i].text(0.97, 0.95, f'Std: {std:.2f}%', transform=axes[i].transAxes,
                 ha='right', va='top', fontsize=9,
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.6))
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Daily Return Distributions — Training Data (2022–2024)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Volume Analysis

Trading volume over time — sustained volume drops can indicate data issues or low-liquidity periods.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
axes = axes.flatten()

for i, (ticker, df) in enumerate(etf_data.items()):
    axes[i].bar(df.index, df['Volume'], color=colors[i], alpha=0.6, width=1)
    axes[i].set_title(f'{ticker} — Daily Volume')
    axes[i].set_ylabel('Volume')
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    axes[i].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=30, ha='right')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Trading Volume — Training Data (2022–2024)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Engineer Features

Calls `engineer_features()` from `features.py` to add MA20, MA50, RSI, MACD, and other indicators to each ETF dataframe.

In [ ]:
featured_data = {}

for ticker, df in etf_data.items():
    featured = engineer_features(df, macro)
    featured_data[ticker] = featured
    print(f'{ticker}: {len(featured)} rows after feature engineering (dropped NaN warm-up rows)')

print(f'\nFinal feature columns ({len(featured_data[TICKERS[0]].columns)}):')
print(list(featured_data[TICKERS[0]].columns))

## 7. Visualise Technical Indicators — IVV.AX

A detailed look at the engineered indicators for IVV.AX.

In [ ]:
df_plot = featured_data['IVV.AX']

fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

# Close + moving averages
axes[0].plot(df_plot.index, df_plot['Close'], label='Close', color='#2196F3', linewidth=1.2)
axes[0].plot(df_plot.index, df_plot['MA20'],  label='MA20',  color='#FF9800', linewidth=1, linestyle='--')
axes[0].plot(df_plot.index, df_plot['MA50'],  label='MA50',  color='#E53935', linewidth=1, linestyle='--')
axes[0].set_title('IVV.AX — Close Price with Moving Averages')
axes[0].set_ylabel('Price (AUD)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(df_plot.index, df_plot['RSI'], color='#9C27B0', linewidth=1)
axes[1].axhline(70, color='red',   linestyle='--', linewidth=0.8, label='Overbought (70)')
axes[1].axhline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold (30)')
axes[1].fill_between(df_plot.index, df_plot['RSI'], 70,
                     where=(df_plot['RSI'] >= 70), alpha=0.15, color='red')
axes[1].fill_between(df_plot.index, df_plot['RSI'], 30,
                     where=(df_plot['RSI'] <= 30), alpha=0.15, color='green')
axes[1].set_title('RSI (14-day)')
axes[1].set_ylabel('RSI')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# MACD
axes[2].plot(df_plot.index, df_plot['MACD'],        label='MACD',   color='#2196F3', linewidth=1)
axes[2].plot(df_plot.index, df_plot['MACD_Signal'], label='Signal', color='#FF9800', linewidth=1, linestyle='--')
axes[2].bar(df_plot.index, df_plot['MACD'] - df_plot['MACD_Signal'],
            color=np.where(df_plot['MACD'] >= df_plot['MACD_Signal'], '#4CAF50', '#E53935'),
            alpha=0.4, width=1, label='Histogram')
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_title('MACD')
axes[2].set_ylabel('Value')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

# Intraday range
axes[3].fill_between(df_plot.index, df_plot['Intraday_Range'], alpha=0.5, color='#607D8B')
axes[3].set_title('Intraday Range (High − Low)')
axes[3].set_ylabel('AUD')
axes[3].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[3].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(axes[3].xaxis.get_majorticklabels(), rotation=30, ha='right')
axes[3].grid(True, alpha=0.3)

plt.suptitle('IVV.AX — Engineered Technical Indicators', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Feature Correlation Heatmap — IVV.AX

Shows how strongly each feature correlates with the Close price and with each other. Highly correlated features are redundant — this is expected for MA20/MA50 vs Close.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

corr = featured_data['IVV.AX'][FEATURES].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8)

ax.set_xticks(range(len(FEATURES)))
ax.set_yticks(range(len(FEATURES)))
ax.set_xticklabels(FEATURES, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(FEATURES, fontsize=9)

for i in range(len(FEATURES)):
    for j in range(len(FEATURES)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                fontsize=7, color='black')

ax.set_title('Feature Correlation Matrix — IVV.AX', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

## 9. Scale Features

MinMaxScaler normalises all features to [0, 1]. The scaler is fitted on training data only — the same scaler is later applied to 2025 test data in `evaluate.py`.

In [ ]:
from model import prepare_sequences, split_sequences

sequence_data = {}

for ticker in TICKERS:
    df = featured_data[ticker]
    X, y, scaler = prepare_sequences(df)
    X_train, X_val, y_train, y_val = split_sequences(X, y)
    sequence_data[ticker] = {
        'X_train': X_train, 'X_val': X_val,
        'y_train': y_train, 'y_val': y_val,
        'scaler': scaler
    }
    print(f'{ticker}:')
    print(f'  X_train: {X_train.shape}  |  X_val: {X_val.shape}')
    print(f'  y_train: {y_train.shape}  |  y_val: {y_val.shape}')

## 10. Visualise a Single Sequence

Each input to the LSTM is a 60-day window of all 18 features. Here is one example sequence for IVV.AX.

In [ ]:
example_seq = sequence_data['IVV.AX']['X_train'][0]  # shape: (60, 18)

fig, ax = plt.subplots(figsize=(14, 5))
for i, feature in enumerate(FEATURES):
    ax.plot(example_seq[:, i], linewidth=0.9, alpha=0.7, label=feature)

ax.set_title('Example Input Sequence — IVV.AX (60 days, all features scaled to [0,1])')
ax.set_xlabel('Day in sequence')
ax.set_ylabel('Scaled value')
ax.legend(fontsize=7, ncol=4, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Sequence shape: {example_seq.shape} → (timesteps, features)')

## 11. Train / Validation Split Visualisation

The chronological 90/10 split shown on the raw closing price — confirms no future data leaks into training.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, ticker in enumerate(TICKERS):
    df = featured_data[ticker]
    split_idx = int(len(df) * 0.9)
    train_df = df.iloc[:split_idx]
    val_df   = df.iloc[split_idx:]

    axes[i].plot(train_df.index, train_df['Close'], color=colors[i], linewidth=1.2, label='Training')
    axes[i].plot(val_df.index,   val_df['Close'],   color='gray',     linewidth=1.2, label='Validation', linestyle='--')
    axes[i].axvline(train_df.index[-1], color='black', linestyle=':', linewidth=1)
    axes[i].set_title(f'{ticker} — Train / Validation Split')
    axes[i].set_ylabel('Price (AUD)')
    axes[i].legend(fontsize=9)
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    axes[i].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=30, ha='right')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Chronological Train / Validation Split (90% / 10%)', fontsize=13)
plt.tight_layout()
plt.show()

## 12. Save Processed Data

In [ ]:
os.makedirs('data/processed', exist_ok=True)

for ticker, df in featured_data.items():
    filename = f"data/processed/{ticker.replace('.', '_')}_featured.csv"
    df.to_csv(filename)
    print(f'Saved {filename}')

print('\nAll processed data saved to data/processed/')

## Summary

| | |
|---|---|
| Features engineered | MA20, MA50, Daily Return, Volume Delta, RSI, MACD, MACD Signal, Intraday Range |
| Total features | 18 (8 technical + 5 macro + 5 OHLCV) |
| Sequence length | 60 days |
| Split | 90% train / 10% validation (chronological) |
| Processed data saved | `data/processed/` |

**Next → `03_model_training.ipynb`** — build and train the LSTM for all four ETFs.